# Guía L2 — El partido que se acumula · **Sesión 1: acumular**

**Cálculo Integral — Laboratorio computacional** · Universidad Nacional de Colombia, Sede Manizales

> Hay una gráfica que ya han visto mil veces abajo en la pantalla de una transmisión: el **momentum**. Hoy la tratamos como lo que matemáticamente es — una tasa con signo — y la acumulamos para medir cuánto partido se llevó cada equipo en la final del Mundial 2022. La historia completa está en la guía impresa; léanla antes de tocar este cuaderno, porque este cuaderno es la mitad ejecutable de esa historia.

**Cómo usar este cuaderno en Colab:** primero guarden su propia copia (`Archivo → Guardar una copia en Drive`) y renómbrenla `L2_apellidos.ipynb`. Trabajen sobre la copia. Las celdas marcadas con ✏️ son de ustedes: hay que completarlas. Las celdas de texto marcadas con 🗣️ son para escribir conclusiones del equipo, con sus palabras.

---
## ⏸️ Antes de ejecutar nada: Momento 1 — Predecir

Está en la guía (P1, P2, P3), con la gráfica impresa del momentum de la final. **En papel, a mano, 15 minutos.** Cuando terminen, tómenle foto: va en el informe. Si ya lo hicieron, sigan. Si no, este cuaderno puede esperar; la predicción no.

---
## 1 · Cargar y conocer

El archivo `momentum_final2022.csv` es la tasa de dominio **del curso**, minuto a minuto, producida con `preparar_momentum.py` a partir del evento a evento de StatsBomb. Columnas:

- `minuto`: minuto **corrido** (tiempo real de juego, con descuentos; prórroga incluida, penales excluidos).
- `peligro_arg`, `peligro_fra`: puntos de peligro que generó cada equipo en ese minuto.
- `momentum_bruto`: la diferencia `peligro_arg − peligro_fra`.
- `momentum`: la diferencia **suavizada** (con memoria de 3 minutos). Esta es nuestra $m$. Unidades: *puntos de peligro por minuto*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/momentum_final2022.csv"
df = pd.read_csv(RUTA)

print("Filas y columnas:", df.shape)
df.head(10)

In [ ]:
# ✏️ Ejercicio 1 — Completen.
print("Primer minuto:", df["minuto"].min())
# print("Último minuto:", ...)            # ← su turno
# print("¿Hay minutos repetidos o saltados?:", ...)   # pista: compárenlo con np.arange(len(df))

# 🗣️ Un partido de "90 minutos más 30 de prórroga" tiene aquí 142 minutos. ¿Por qué?
#    (Pista: el reloj oficial se detiene al final de cada tiempo; el minuto corrido no.)

---
## 2 · El dominio de cada columna

El ritual del laboratorio 1, con una novedad: allá un consumo negativo era **ilegal**; aquí los negativos son **el corazón del dato** — significan que domina Francia.

🗣️ **Completen esta tabla (doble clic para editar):**

| Columna | Dominio matemático | ¿Puede ser negativa? ¿Por qué? |
|---|---|---|
| `minuto` | | |
| `peligro_arg` | | |
| `peligro_fra` | | |
| `momentum` | | |

In [ ]:
# La policía de dominios.
print(df.describe().round(3))
print()
print("NaN por columna:\n", df.isna().sum())
print()
print("¿Peligro negativo en Argentina?:", (df["peligro_arg"] < 0).sum())
print("¿Peligro negativo en Francia?:  ", (df["peligro_fra"] < 0).sum())
print("Minutos con momentum negativo:  ", (df["momentum"] < 0).sum(), "← no son errores: son minutos de Francia")

# ✏️ Ejercicio 2 — Una columna que afirma ser una resta, se audita.
# Verifiquen que momentum_bruto == peligro_arg − peligro_fra en TODAS las filas.
# print("¿La resta cuadra?:", np.allclose(..., ...))   # ← su turno

---
## 3 · La escalera: el partido como función escalonada

Partición: $P = \{t_0, t_1, \ldots, t_{142}\} = \{0, 1, \ldots, 142\}$. Regla: $m(t) = m_k$ si $k < t < k+1$. Durante el minuto $k$, el dominio corre a razón de $m_k$ puntos/min.

Los tiempos del partido, en minutos corridos: primer tiempo $[0,53]$, segundo $[53,106]$, primera prórroga $[106,122]$, segunda $[122,142]$.

In [ ]:
m = df["momentum"].to_numpy()          # alturas: m_0, ..., m_141
t = np.arange(len(m) + 1)              # partición: 0, 1, ..., 142
CORTES = [0, 53, 106, 122, 142]        # fronteras de los tiempos

m_esc = np.append(m, m[-1])            # truco de graficación: el último escalón también se dibuja
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(t, np.maximum(m_esc, 0), 0, step="post", color="#2F6DB5", alpha=0.6, label="domina Argentina (+)")
ax.fill_between(t, np.minimum(m_esc, 0), 0, step="post", color="#C9573C", alpha=0.6, label="domina Francia (−)")
ax.step(t, m_esc, where="post", color="black", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.8)
for c in CORTES[1:-1]:
    ax.axvline(c, color="gray", linestyle="--")
ax.set_xlabel("Minuto corrido"); ax.set_ylabel("m(t)  (puntos de peligro / min)")
ax.set_title("La tasa de dominio de la final 2022: una función escalonada con signo")
ax.legend(); plt.show()

🗣️ **El modelo, por escrito (doble clic):** la partición $P$, los primeros tres valores $m_0, m_1, m_2$ con sus unidades, y la regla completa de $m(t)$. ¿Qué vale $m$ en $t = 53$ exactamente, la frontera entre el primer y el segundo tiempo? ¿Le importa al dominio del partido?

---
## 4 · El producto, con unidades

Antes de sumar nada: ¿qué **es** $m_{65} \cdot (t_{66} - t_{65})$? ¿Y qué significa ese producto cuando $m_k < 0$?

In [ ]:
k = 65
producto = m[k] * (t[k+1] - t[k])
print(f"m_{k} = {m[k]} puntos/min   (la altura: el ritmo de dominio en ese minuto)")
print(f"Δt = {t[k+1] - t[k]} min           (la base: lo que dura ese ritmo)")
print(f"m_{k} · Δt = {producto:.4f} puntos   ← dominio aportado por el minuto {k}")

# ✏️ Ejercicio 3 — Busquen un minuto con m_k < 0 e impriman su producto.
# 🗣️ Escríbanlo con palabras: "en el minuto ___ dominó ___ a razón de ___ puntos/min durante 1 min,
#    luego aportó ___ puntos". ¿A quién se los aportó? El signo también es información.

---
## 5 · La integral con signo, tres veces

$$\int_0^{142} m(t)\,dt \;=\; \sum_{k=0}^{141} m_k\,(t_{k+1} - t_k)$$

Las tres formas del laboratorio 1 deben coincidir hasta el último decimal.

In [ ]:
# (a) un ciclo, término a término
neto_a = 0.0
for k in range(len(m)):
    neto_a = neto_a + m[k] * (t[k+1] - t[k])

# (b) vectorizado
neto_b = np.sum(m * np.diff(t))

# (c) el área de la escalera, rectángulo por rectángulo
colores = np.where(m >= 0, "#2F6DB5", "#C9573C")
plt.figure(figsize=(12, 4))
plt.bar(t[:-1], m, width=1.0, align="edge", color=colores, alpha=0.7)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Minuto corrido"); plt.ylabel("puntos/min")
plt.title(f"∫ m(t) dt = {neto_b:.4f} puntos de peligro")
plt.show()

print(f"(a) ciclo:   {neto_a:.6f}")
print(f"(b) np.sum:  {neto_b:.6f}")
print(f"¿Coinciden?  {np.isclose(neto_a, neto_b)}")

In [ ]:
# ✏️ Ejercicio 4 — La integral NO es un área: es una resta de áreas.
area_azul = np.sum(m[m > 0])          # todo lo de arriba
# area_roja = ...                     # ← su turno: lo de abajo, como número POSITIVO
# print(f"área azul − área roja = {area_azul - area_roja:.4f}   vs   ∫m = {neto_b:.4f}")

# 🗣️ Confronten P1: ¿quién acumuló más partido? ¿Quién del equipo tuvo el mejor ojo?
#    ¿Alguien estimó el área total (azul + roja) en vez del neto? ¿Qué mediría esa otra cantidad?

---
## 6 · La función de acumulación $D(x)$

$$D(x) = \int_0^x m(t)\,dt, \qquad x = 0, 1, \ldots, 142$$

Ya no es **un** número: es una **función** de su límite superior. Para cada minuto $x$, $D(x)$ dice cuánto partido llevaba acumulado Argentina hasta ese momento.

In [ ]:
D = np.concatenate([[0.0], np.cumsum(m * np.diff(t))])   # D(0) = 0: ¿por qué tiene que empezar en cero?
print("Valores de D:", len(D), "  (uno más que los escalones: ¿por qué?)")
print("D(142) =", round(D[-1], 4), " ← ¿qué número es este? Compárenlo con la sección 5.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.fill_between(t, np.maximum(m_esc, 0), 0, step="post", color="#2F6DB5", alpha=0.6)
ax1.fill_between(t, np.minimum(m_esc, 0), 0, step="post", color="#C9573C", alpha=0.6)
ax1.axhline(0, color="black", linewidth=0.8); ax1.set_ylabel("m(t)  [puntos/min]")
ax1.set_title("Arriba la tasa, abajo lo acumulado")
ax2.plot(t, D, color="black", linewidth=2)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("D(x)  [puntos]"); ax2.set_xlabel("Minuto corrido")
for ax in (ax1, ax2):
    for c in CORTES[1:-1]:
        ax.axvline(c, color="gray", linestyle="--")
plt.show()

In [ ]:
# ✏️ Ejercicio 5 — P2: dos preguntas que parecían la misma.
k_tasa = int(np.argmax(m))     # el minuto del mayor apretón (máximo de la TASA)
# x_acum = ...                 # ← su turno: el minuto del partido más ganado (máximo de la ACUMULADA)
print("Máximo de la tasa m:", round(m[k_tasa], 3), "en el minuto", k_tasa)
# print("Máximo de la acumulada D:", ..., "en x =", ...)

# ¿Qué está haciendo m alrededor del máximo de D? Miren los escalones vecinos:
# print(m[x_acum-3 : x_acum+3])

# 🗣️ ¿Coinciden los dos minutos? ¿Qué le pasa al SIGNO de m justo donde D alcanza su máximo?
#    Escriban la regla que acaban de descubrir (en el tablero tiene nombre: punto crítico).

---
## 7 · Los picos gemelos

Miren la gráfica de $D$ con calma: tiene **dos** picos casi de la misma altura, uno en el tiempo reglamentario y otro en la prórroga.

In [ ]:
# ✏️ Ejercicio 6 — Encuentren los dos picos.
pico_reglamentario = int(np.argmax(D[:107]))     # el máximo de D en [0, 106]
# pico_prorroga = ...                            # ← su turno: el máximo de D en [106, 142]
print("Pico en el tiempo reglamentario: D =", round(D[pico_reglamentario], 2), "en x =", pico_reglamentario)
# print("Pico en la prórroga:             D =", ..., "en x =", ...)

# 🗣️ ¿Qué pasó en la cancha pocos minutos después de cada pico? (Miren su memoria del partido,
#    o adelanten la sesión 2: los goles están en eventos_final2022.csv, columna t_corrido.)
#    Tres líneas: ¿qué cuenta D que el marcador no cuenta?

---
## 8 · Aditividad por tiempos

$$\int_0^{142} m = \int_0^{53} m + \int_{53}^{106} m + \int_{106}^{122} m + \int_{122}^{142} m$$

Con $D$ en la mano, cada pedazo es una resta: $\int_a^b m = D(b) - D(a)$. ¿Por qué?

In [ ]:
nombres = ["1.er tiempo", "2.º tiempo", "prórroga 1", "prórroga 2"]
total = 0.0
for nombre, a, b in zip(nombres, CORTES[:-1], CORTES[1:]):
    pedazo = D[b] - D[a]
    total = total + pedazo
    print(f"{nombre:12s} ∫_{a}^{b} m = {pedazo:+8.3f} puntos")
    # ✏️ Ejercicio 7 — agreguen aquí el valor promedio del tiempo: (1/(b−a)) ∫_a^b m, en puntos/min
print(f"Suma de los pedazos: {total:.4f}   vs   D(142) = {D[-1]:.4f}")

# 🗣️ Los tiempos duran distinto (53, 53, 16 y 20 minutos). Comparar sus integrales sin normalizar
#    sería el misterio de febrero otra vez. Según el valor promedio, ¿qué tiempo fue de Francia?
#    ¿Cuál fue el tiempo más argentino: el de mayor integral o el de mayor promedio?

---
## 9 · Derivar lo acumulado devuelve la tasa

El laboratorio 1 terminó con una pregunta: en la curva del contador, ¿qué información quedó escondida en la **pendiente**? Hoy se responde.

In [ ]:
pendientes = np.diff(D)           # D(x+1) − D(x), minuto a minuto
print("Primeras pendientes de D:", np.round(pendientes[:6], 4))
print("Primeros escalones de m: ", np.round(m[:6], 4))
print("¿Son exactamente iguales en los 142 minutos?:", np.allclose(pendientes, m))

# ✏️ Ejercicio 8 — Dentro de un minuto, la pendiente es exacta, no aproximada.
# Tomen x = 125.3 y h = 0.4 (ambos dentro del minuto 125). Calculen D(x) a mano con la regla
# D(x) = D(⌊x⌋) + m_⌊x⌋ · (x − ⌊x⌋) y verifiquen que (D(x+h) − D(x)) / h es EXACTAMENTE m_125.
# def D_continua(x): ...          # ← su turno

# 🗣️ La pendiente de la curva del contador era el consumo; la pendiente de D es ______.
#    Esto es el Teorema Fundamental del Cálculo que vieron en clase, en versión escalonada:
#    D'(x) = m(x) en todo x que no sea entero. ¿Por qué falla justo en los enteros?

---
## 🗣️ Momento 3 — Explicar

Media página del equipo, aquí mismo: ¿qué predijeron (P1, P2, P3)? ¿Qué encontraron? ¿Distinguieron la tasa de la acumulada al primer intento?

**Frase obligatoria del informe — complétenla con sus palabras:** “Una función de acumulación responde preguntas que la tasa sola no puede responder, por ejemplo…”

---
## 🤖 Protocolo de IA (recordatorio)

La IA es tutora y auditora, no autora. Intento propio antes de consultar; la IA no es la autoridad final; **todo uso va a la bitácora individual** (prompt → fragmento de respuesta → qué aceptaron / corrigieron / rechazaron). Para depurar: *"Este código da este error: [ambos]. No me des el código corregido; explícame el error y hazme una pregunta que me guíe."* Todo integrante debe poder explicar cualquier celda si el monitor la señala.

**Esta semana: la tarea puente** (está en la guía): correr `preparar_momentum.py` contra la fuente y reproducir este CSV exacto, defender o atacar la exclusión de los penales, y juzgar los goles que pesan raro. **Próxima sesión:** reciben el evento a evento crudo y fabrican **su propio** momentum.